# Homework 5.1 - Step 1: Source Corpus Preparation

This notebook prepares the two public-domain English corpora required for Homework 5.1.

The goal of this step is to:

1. identify and cite both source editions,
2. download the complete texts,
3. remove non-primary front and back matter,
4. extract every required book using the actual book headings,
5. normalize the text consistently,
6. save the cleaned corpora for Step 2.

Syllable tokenization is intentionally left for Step 2.

## Sources

### Marcus Aurelius, *Meditations*

- Author: Marcus Aurelius
- English translation: Meric Casaubon
- Project Gutenberg eBook: #2680
- Source: [Project Gutenberg - Meditations](https://www.gutenberg.org/cache/epub/2680/pg2680-images.html)

### Plato, *The Republic*

- Author: Plato
- Translator: Benjamin Jowett
- Project Gutenberg eBook: #1497
- Source: [Project Gutenberg - The Republic](https://www.gutenberg.org/cache/epub/1497/pg1497-images.html)

These are the exact Gutenberg sources used for the corpus preparation.

## Cleaning convention

For both corpora:

- keep only the primary text,
- remove Gutenberg boilerplate and editorial front or back matter,
- preserve ordinary punctuation,
- preserve ordinary word boundaries,
- convert curly quotation marks and apostrophes to plain ASCII forms,
- convert long dashes to a normal hyphen,
- normalize repeated whitespace,
- convert the final corpus text to lowercase.

For *Meditations*, all twelve books are retained.

For *The Republic*, all ten books of the dialogue are retained.

The book headings are used to identify and verify the extraction boundaries. The headings themselves are not included in the final cleaned corpus text because they are structural labels rather than part of the linguistic content to be syllabified.

In [8]:
from pathlib import Path
import csv
import hashlib
import re
import unicodedata

import requests
from bs4 import BeautifulSoup

# If this notebook is placed inside codes/, output goes to the sibling data/ folder.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MEDITATIONS_URL = "https://www.gutenberg.org/cache/epub/2680/pg2680-images.html"
REPUBLIC_URL = "https://www.gutenberg.org/cache/epub/1497/pg1497-images.html"

print("Project root:", PROJECT_ROOT.resolve())
print("Data folder:", DATA_DIR.resolve())

Project root: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1
Data folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data


## Download the two Gutenberg HTML pages

In [9]:
def download_html(url, output_path):
    response = requests.get(
        url,
        timeout=60,
        headers={"User-Agent": "Math638-HW5.1-corpus-preparation"},
    )
    response.raise_for_status()

    html = response.text
    output_path.write_text(html, encoding="utf-8")
    return html


meditations_html = download_html(
    MEDITATIONS_URL,
    DATA_DIR / "meditations_raw.html",
)

republic_html = download_html(
    REPUBLIC_URL,
    DATA_DIR / "republic_raw.html",
)

print("Downloaded both Project Gutenberg sources.")

Downloaded both Project Gutenberg sources.


## Helper functions

In [10]:
CHAR_MAP = str.maketrans({
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u2013": "-",
    "\u2014": "-",
    "\u00a0": " ",
})


def normalize_marker(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(CHAR_MAP)
    text = re.sub(r"\s+", " ", text).strip()
    return text.upper()


def normalize_paragraph(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(CHAR_MAP)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def visible_text(tag):
    temp = BeautifulSoup(str(tag), "html.parser")

    for sup in temp.find_all("sup"):
        sup.decompose()

    return temp.get_text(" ", strip=True)


def word_count(text):
    return len(re.findall(r"[a-z]+(?:'[a-z]+)?", text))


def corpus_stats(text):
    return {
        "characters": len(text),
        "words": word_count(text),
        "paragraphs": len([p for p in text.split("\n\n") if p.strip()]),
    }


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def strip_leading_roman_numeral(text):
    # Example:
    # "I. Of my grandfather..." -> "Of my grandfather..."
    return re.sub(r"^\s*[IVXLCDM]+\.\s*", "", text).strip()

## Extract all twelve books of *Meditations*

The Gutenberg page contains the book headings throughout the primary text.

The first book is identified by **THE FIRST BOOK**, followed by **THE SECOND BOOK**, continuing through **THE TWELFTH BOOK**.

Each book is extracted separately so that the complete corpus can be checked before combining the books.

In [11]:
meditations_book_headings = [
    "THE FIRST BOOK",
    "THE SECOND BOOK",
    "THE THIRD BOOK",
    "THE FOURTH BOOK",
    "THE FIFTH BOOK",
    "THE SIXTH BOOK",
    "THE SEVENTH BOOK",
    "THE EIGHTH BOOK",
    "THE NINTH BOOK",
    "THE TENTH BOOK",
    "THE ELEVENTH BOOK",
    "THE TWELFTH BOOK",
]

med_soup = BeautifulSoup(meditations_html, "html.parser")

# Search heading tags only so table-of-contents hyperlinks are ignored.
meditation_book_tags = {}

for heading in meditations_book_headings:
    target = heading.upper()
    matches = []

    for tag in med_soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        if normalize_marker(tag.get_text(" ", strip=True)) == target:
            matches.append(tag)

    if len(matches) == 0:
        raise ValueError(f"Could not find heading: {heading}")

    # If Gutenberg contains more than one matching heading,
    # use the last occurrence to avoid a front-matter entry.
    meditation_book_tags[heading] = matches[-1]

print("Found all Meditations book headings:")
for heading in meditations_book_headings:
    print(heading)

Found all Meditations book headings:
THE FIRST BOOK
THE SECOND BOOK
THE THIRD BOOK
THE FOURTH BOOK
THE FIFTH BOOK
THE SIXTH BOOK
THE SEVENTH BOOK
THE EIGHTH BOOK
THE NINTH BOOK
THE TENTH BOOK
THE ELEVENTH BOOK
THE TWELFTH BOOK


In [12]:
meditation_books = {}

for i, heading in enumerate(meditations_book_headings):
    start_tag = meditation_book_tags[heading]

    if i < len(meditations_book_headings) - 1:
        next_heading = meditations_book_headings[i + 1]
        stop_tag = meditation_book_tags[next_heading]
    else:
        stop_tag = None

    paragraphs = []

    for tag in start_tag.find_all_next():

        # Stop when the next Meditations book begins.
        if stop_tag is not None and tag is stop_tag:
            break

        tag_text = tag.get_text(" ", strip=True)

        # For Book XII, stop when the Appendix begins.
        if stop_tag is None and normalize_marker(tag_text) == "APPENDIX":
            break

        if tag.name == "p":
            temp = BeautifulSoup(str(tag), "html.parser")

            for sup in temp.find_all("sup"):
                sup.decompose()

            text = temp.get_text(" ", strip=True)
            text = strip_leading_roman_numeral(text)
            text = normalize_paragraph(text)

            if text:
                paragraphs.append(text)

    book_text = "\n\n".join(paragraphs).strip()

    if not book_text:
        raise ValueError(f"No text extracted for {heading}")

    meditation_books[i + 1] = book_text


print(f"Successfully extracted {len(meditation_books)} Meditations books.")

Successfully extracted 12 Meditations books.


### Verify all twelve *Meditations* books

In [13]:
print(f"{'Book':<8}{'Characters':>15}{'Words':>12}{'Paragraphs':>14}")
print("-" * 49)

meditation_summary = []

for book_number in range(1, 13):
    text = meditation_books[book_number]
    stats = corpus_stats(text)

    meditation_summary.append({
        "corpus": "Meditations",
        "book": book_number,
        **stats,
    })

    print(
        f"{book_number:<8}"
        f"{stats['characters']:>15,}"
        f"{stats['words']:>12,}"
        f"{stats['paragraphs']:>14,}"
    )

Book         Characters       Words    Paragraphs
-------------------------------------------------
1                21,109       3,801            17
2                11,921       2,203            16
3                18,377       3,308            17
4                29,735       5,361            43
5                29,675       5,359            31
6                30,212       5,520            52
7                32,558       5,937            45
8                31,802       5,835            57
9                28,972       5,297            43
10               33,547       6,183            38
11               24,522       4,446            31
12               20,180       3,691            27


## Extract all ten books of *The Republic*

The primary dialogue is identified by the actual headings

**BOOK I.**, **BOOK II.**, ..., **BOOK X.**

The table-of-contents links are ignored because only heading tags are searched.

Books I through IX end when the next book heading begins. Book X ends at Gutenberg's end-of-ebook marker.

In [14]:
republic_book_headings = [
    "BOOK I.",
    "BOOK II.",
    "BOOK III.",
    "BOOK IV.",
    "BOOK V.",
    "BOOK VI.",
    "BOOK VII.",
    "BOOK VIII.",
    "BOOK IX.",
    "BOOK X.",
]

rep_soup = BeautifulSoup(republic_html, "html.parser")

republic_book_tags = {}

for heading in republic_book_headings:
    target = heading.upper()
    matches = []

    for tag in rep_soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        if normalize_marker(tag.get_text(" ", strip=True)) == target:
            matches.append(tag)

    if len(matches) == 0:
        raise ValueError(f"Could not find heading: {heading}")

    # Use the last match so an earlier table-of-contents or front-matter
    # occurrence does not get used accidentally.
    republic_book_tags[heading] = matches[-1]

print("Found all Republic book headings:")
for heading in republic_book_headings:
    print(heading)

Found all Republic book headings:
BOOK I.
BOOK II.
BOOK III.
BOOK IV.
BOOK V.
BOOK VI.
BOOK VII.
BOOK VIII.
BOOK IX.
BOOK X.


In [15]:
republic_books = {}

for i, heading in enumerate(republic_book_headings):
    start_tag = republic_book_tags[heading]

    if i < len(republic_book_headings) - 1:
        next_heading = republic_book_headings[i + 1]
        stop_tag = republic_book_tags[next_heading]
    else:
        stop_tag = None

    paragraphs = []

    for tag in start_tag.find_all_next():

        # Books I-IX stop at the next book heading.
        if stop_tag is not None and tag is stop_tag:
            break

        tag_text = tag.get_text(" ", strip=True)

        # Book X stops at the Gutenberg end marker.
        if (
            stop_tag is None
            and "*** END OF THE PROJECT GUTENBERG EBOOK THE REPUBLIC ***"
            in tag_text.upper()
        ):
            break

        if tag.name == "p":
            temp = BeautifulSoup(str(tag), "html.parser")

            for sup in temp.find_all("sup"):
                sup.decompose()

            text = temp.get_text(" ", strip=True)
            text = normalize_paragraph(text)

            if text:
                paragraphs.append(text)

    book_text = "\n\n".join(paragraphs).strip()

    if not book_text:
        raise ValueError(f"No text extracted for {heading}")

    republic_books[i + 1] = book_text


print(f"Successfully extracted {len(republic_books)} Republic books.")

Successfully extracted 10 Republic books.


### Verify all ten *Republic* books

In [16]:
print(f"{'Book':<8}{'Characters':>15}{'Words':>12}{'Paragraphs':>14}")
print("-" * 49)

republic_summary = []

for book_number in range(1, 11):
    text = republic_books[book_number]
    stats = corpus_stats(text)

    republic_summary.append({
        "corpus": "The Republic",
        "book": book_number,
        **stats,
    })

    print(
        f"{book_number:<8}"
        f"{stats['characters']:>15,}"
        f"{stats['words']:>12,}"
        f"{stats['paragraphs']:>14,}"
    )

Book         Characters       Words    Paragraphs
-------------------------------------------------
1                65,304      12,383           492
2                62,297      11,612           349
3                75,542      13,961           497
4                62,023      11,468           457
5                75,317      14,007           563
6                64,592      11,904           408
7                62,958      11,579           372
8                61,845      11,311           447
9                48,663       8,985           405
10               62,835      11,701           318


## Save each book separately

Saving each book separately provides a direct verification that all twelve *Meditations* books and all ten *Republic* books were extracted.

In [17]:
MED_BOOK_DIR = DATA_DIR / "meditations_books"
REP_BOOK_DIR = DATA_DIR / "republic_books"

MED_BOOK_DIR.mkdir(exist_ok=True)
REP_BOOK_DIR.mkdir(exist_ok=True)

for book_number, text in meditation_books.items():
    path = MED_BOOK_DIR / f"book_{book_number:02d}.txt"
    path.write_text(text + "\n", encoding="utf-8")

for book_number, text in republic_books.items():
    path = REP_BOOK_DIR / f"book_{book_number:02d}.txt"
    path.write_text(text + "\n", encoding="utf-8")

print("Saved 12 Meditations book files.")
print("Saved 10 Republic book files.")

Saved 12 Meditations book files.
Saved 10 Republic book files.


## Combine all books into the two cleaned corpora

In [18]:
meditations_clean = "\n\n".join(
    meditation_books[i]
    for i in range(1, 13)
).strip() + "\n"

republic_clean = "\n\n".join(
    republic_books[i]
    for i in range(1, 11)
).strip() + "\n"

meditations_clean_path = DATA_DIR / "meditations_cleaned.txt"
republic_clean_path = DATA_DIR / "republic_cleaned.txt"

meditations_clean_path.write_text(
    meditations_clean,
    encoding="utf-8",
)

republic_clean_path.write_text(
    republic_clean,
    encoding="utf-8",
)

print("Saved complete Meditations corpus:")
print(meditations_clean_path.resolve())

print()

print("Saved complete Republic corpus:")
print(republic_clean_path.resolve())

Saved complete Meditations corpus:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\meditations_cleaned.txt

Saved complete Republic corpus:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\republic_cleaned.txt


## Save book-by-book extraction statistics

In [19]:
book_summary_rows = meditation_summary + republic_summary

book_summary_path = DATA_DIR / "book_extraction_summary.csv"

with book_summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "corpus",
            "book",
            "characters",
            "words",
            "paragraphs",
        ],
    )

    writer.writeheader()
    writer.writerows(book_summary_rows)

print("Saved:", book_summary_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\book_extraction_summary.csv


## Final corpus metadata

In [20]:
med_stats = corpus_stats(meditations_clean)
rep_stats = corpus_stats(republic_clean)

metadata_rows = [
    {
        "corpus": "Meditations",
        "author": "Marcus Aurelius",
        "translator": "Meric Casaubon",
        "gutenberg_ebook": "2680",
        "source_url": MEDITATIONS_URL,
        "books_retained": 12,
        "characters": med_stats["characters"],
        "words": med_stats["words"],
        "paragraphs": med_stats["paragraphs"],
        "sha256": sha256_text(meditations_clean),
    },
    {
        "corpus": "The Republic",
        "author": "Plato",
        "translator": "Benjamin Jowett",
        "gutenberg_ebook": "1497",
        "source_url": REPUBLIC_URL,
        "books_retained": 10,
        "characters": rep_stats["characters"],
        "words": rep_stats["words"],
        "paragraphs": rep_stats["paragraphs"],
        "sha256": sha256_text(republic_clean),
    },
]

metadata_path = DATA_DIR / "corpus_metadata.csv"

with metadata_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=metadata_rows[0].keys(),
    )

    writer.writeheader()
    writer.writerows(metadata_rows)

for row in metadata_rows:
    print()
    print(row["corpus"])
    print("  Books retained:", row["books_retained"])
    print("  Characters:", f'{row["characters"]:,}')
    print("  Words:", f'{row["words"]:,}')
    print("  Paragraphs:", f'{row["paragraphs"]:,}')
    print("  SHA-256:", row["sha256"])

print()
print("Saved:", metadata_path.resolve())


Meditations
  Books retained: 12
  Characters: 312,633
  Words: 56,941
  Paragraphs: 417
  SHA-256: 4317d3565d43d4254a1623f8e961eea8f79418bd0b4d599db0343e8b0f03db73

The Republic
  Books retained: 10
  Characters: 641,395
  Words: 118,911
  Paragraphs: 4,308
  SHA-256: 80342c7392293a542535c36ddfbb0aac78ee43d581a29302379e02b68e97e090

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\corpus_metadata.csv


## Final completeness check

Instead of showing only the beginning of each corpus, this cell prints:

- the beginning of *Meditations* Book I,
- the end of *Meditations* Book XII,
- the beginning of *Republic* Book I,
- the end of *Republic* Book X.

This provides a simple final check that the extraction did not stop after the first book.

In [21]:
print("MEDITATIONS - BEGINNING OF BOOK I")
print("-" * 80)
print(meditation_books[1][:700])

print("\n\nMEDITATIONS - END OF BOOK XII")
print("-" * 80)
print(meditation_books[12][-700:])

print("\n\nREPUBLIC - BEGINNING OF BOOK I")
print("-" * 80)
print(republic_books[1][:700])

print("\n\nREPUBLIC - END OF BOOK X")
print("-" * 80)
print(republic_books[10][-700:])

MEDITATIONS - BEGINNING OF BOOK I
--------------------------------------------------------------------------------
of my grandfather verus i have learned to be gentle and meek, and to refrain from all anger and passion. from the fame and memory of him that begot me i have learned both shamefastness and manlike behaviour. of my mother i have learned to be religious, and bountiful; and to forbear, not only to do, but to intend any evil; to content myself with a spare diet, and to fly all such excess as is incidental to great wealth. of my great-grandfather, both to frequent public schools and auditories, and to get me good and able teachers at home; and that i ought not to think much, if upon such occasions, i were at excessive charges.

of him that brought me up, not to be fondly addicted to either of the


MEDITATIONS - END OF BOOK XII
--------------------------------------------------------------------------------
t of all. why then should it be grievous unto thee, if (not a tyrant, n

## Step 1 output

After this notebook is run, the `data/` folder should contain:

- `meditations_raw.html`
- `republic_raw.html`
- `meditations_cleaned.txt`
- `republic_cleaned.txt`
- `book_extraction_summary.csv`
- `corpus_metadata.csv`
- `meditations_books/`
  - `book_01.txt`
  - ...
  - `book_12.txt`
- `republic_books/`
  - `book_01.txt`
  - ...
  - `book_10.txt`

The two cleaned combined text files are the fixed inputs for Step 2.